# This notebook builds a simple FAISS RAG flow.

In [ ]:
# Import the main tools we need.
import os
import re
import json
import hashlib
from typing import Dict, List, Tuple
from pathlib import Path

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from huggingface_hub import login

c:\projects\learn-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the basic libraries.

In [ ]:
# Import more tools and load the env file.
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load values from the root .env file.


True

## Load FAISS, embeddings, and env values.

In [ ]:
# Set the LLM name and start the model.
llm_model_name = "qwen/qwen3-32b"

llm = ChatGroq(
    model=llm_model_name,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
 )

print(f"LLM model: {llm_model_name}")

LLM model: qwen/qwen3-32b


## Set up the chat model.

In [ ]:
# Find the text files in the dataset folder.
datasets_dir = Path(r"C:\projects\learn-rag\Datasets")
txt_files = sorted(datasets_dir.glob("*.txt"))

print(f"Found {len(txt_files)} text file(s) in {datasets_dir}:")
for i, file_path in enumerate(txt_files, start=1):
    print(f"{i}. {file_path.name}")

Found 4 text file(s) in C:\projects\learn-rag\Datasets:
1. amazon_2023.txt
2. amazon_2024.txt
3. microsoft_2023.txt
4. microsoft_2024.txt


## Find the text files we can use.

In [ ]:
# Load the first text file and show a short preview.
if txt_files:
    selected_file = txt_files[0]
    text_data = selected_file.read_text(encoding="utf-8", errors="ignore")
    print(f"\nLoaded file: {selected_file.name}")
    print(f"Character count: {len(text_data):,}")
    print("Preview:\n")
    print(text_data[:500])
else:
    text_data = ""
    print("No .txt files found.")


Loaded file: amazon_2023.txt
Character count: 5,593
Preview:

company_name: Amazon
year: 2023
ticker: AMZN
fiscal_period: FY 2023
headquarters: Seattle, Washington, United States
ceo: Andy Jassy
founded: 1994
industry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI
revenue_usd_billions: 574.7
net_income_usd_billions: 30.4
employee_count: 1525000

Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was found


## Load one file and look at the text.

In [ ]:
# Split the text into small token-based chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=250,
    chunk_overlap=50,
)

docs = splitter.create_documents(
    texts=[text_data],
    metadatas=[{
        "source": str(selected_file),
        "doc_name": selected_file.name,
        "year": "2023",
        "company": "amazon",
    }],
)

docs

[Document(metadata={'source': 'C:\\projects\\learn-rag\\Datasets\\amazon_2023.txt', 'doc_name': 'amazon_2023.txt', 'year': '2023', 'company': 'amazon'}, page_content="company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000\n\nAmazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.\n\nIn 2023, Amazon reported approximately $574.7 billion in revenu

## Split the text into small chunks.

In [ ]:
# Set the embedding model name and load the token.
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

## Choose the embedding model.

In [ ]:
# Load the sentence embedding model.
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(embedding_model_name, device="cpu")

print(f"Loaded embedder: {embedding_model_name}")
print(f"Embedding dimension: {embedder.get_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2635.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedder: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


## Load the embedding model.

In [ ]:
# Make an adapter and turn chunks into vectors.
from langchain_core.embeddings import Embeddings

class SentenceTransformerAdapter(Embeddings):
    # This class helps FAISS use the embedding model.
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0].tolist()

embedding_adapter = SentenceTransformerAdapter(embedder)

chunk_texts = [doc.page_content for doc in docs]
vectors = embedding_adapter.embed_documents(chunk_texts)
print(f"Chunk vectors shape: {len(vectors)} x {len(vectors[0])}")

Chunk vectors shape: 5 x 384


## Turn the chunks into vectors.

In [ ]:
# Show the vectors.
vectors

[[0.00409706961363554,
  -0.07407249510288239,
  -0.022514428943395615,
  -0.013816773891448975,
  0.04362194612622261,
  -0.03130115196108818,
  -0.0035131017211824656,
  0.013085654936730862,
  0.04428891837596893,
  0.03714452311396599,
  -0.029770920053124428,
  0.060666270554065704,
  0.060885537415742874,
  -0.02430819347500801,
  0.007720386143773794,
  -0.039149731397628784,
  0.0042256638407707214,
  -0.1342686265707016,
  -0.07496418803930283,
  -0.12339320778846741,
  0.017941726371645927,
  0.02836916595697403,
  -0.011320268735289574,
  -0.04845317825675011,
  -0.028825121000409126,
  0.03607705608010292,
  -0.04671034589409828,
  -0.015868665650486946,
  -0.03766852617263794,
  -0.04688189551234245,
  0.016512004658579826,
  -0.025698937475681305,
  0.10196054726839066,
  0.05558621138334274,
  -0.043298568576574326,
  0.012452449649572372,
  -0.002036174526438117,
  -0.09347975999116898,
  0.003097549779340625,
  0.005919665563851595,
  0.013442701660096645,
  0.00278604

## View the vector output.

In [ ]:
# Create the FAISS vector store.
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = embedder.get_embedding_dimension()
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embedding_adapter,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

print(f"Vector store initialized with dimension: {embedding_dim}")

Vector store initialized with dimension: 384


## Build the FAISS store.

## Add the vectors into FAISS.

In [ ]:
# Add the saved vectors into the FAISS store.
from uuid_utils import uuid4

uuids = [str(uuid4()) for _ in range(len(vectors))]
text_embeddings = list(zip(chunk_texts, vectors))

vector_store.add_embeddings(text_embeddings=text_embeddings, ids=uuids)
print(f"Added {len(uuids)} embeddings to FAISS")

Added 5 embeddings to FAISS


## Store the vectors in FAISS.

In [ ]:
# Save the FAISS store and load it again.
store_dir = Path.cwd() / "faiss_store"
store_dir.mkdir(parents=True, exist_ok=True)

vector_store.save_local(folder_path=str(store_dir))
print(f"Saved vector store to: {store_dir}")

loaded_vector_store = FAISS.load_local(
    folder_path=str(store_dir),
    embeddings=embedding_adapter,
    allow_dangerous_deserialization=True,
)

Saved vector store to: c:\projects\learn-rag\vectorDB\faiss_store


## Save and reload the FAISS store.

In [ ]:
# Search the store and print the best chunks.
query = "what is revenue of amazon in 2023 ?"
results = vector_store.similarity_search(query, k=20)

print("Top 3 retrieved chunks from loaded store:")
answers = []
for i, doc in enumerate(results, start=1):
    answers.append(doc.page_content)
    print(f"\nResult {i}:\n{doc.page_content[:1000]}")

Top 3 retrieved chunks from loaded store:

Result 1:
company_name: Amazon
year: 2023
ticker: AMZN
fiscal_period: FY 2023
headquarters: Seattle, Washington, United States
ceo: Andy Jassy
founded: 1994
industry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI
revenue_usd_billions: 574.7
net_income_usd_billions: 30.4
employee_count: 1525000

Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.

In 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income. The year reflected a strong recovery in profitability after the weaker 2022 period. Ama

## Search the store and read the results.

## Load the reranker model.

In [ ]:
# Load the model and tokenizer for reranking.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
token_arg = hf_token if hf_token else None

model = AutoModelForSequenceClassification.from_pretrained(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    # Add the token here if the model needs it.
)
tokenizer = AutoTokenizer.from_pretrained(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    # Add the token here if the model needs it.
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1238.99it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Load the reranker files.

In [ ]:
# Start the reranker model.
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', token=token_arg)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2386.99it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Start the reranker model.

In [ ]:
# Rerank the search results and keep the best ones.
import torch

def top_k_rerank(question, answers, top_k=5):
    """Pick the best answers with the reranker."""
    if not answers:
        return []

    # Keep top_k inside the list size.
    top_k = max(1, min(top_k, len(answers)))

    # Build the input pairs for the reranker.
    features = tokenizer(
        [question] * len(answers),
        answers,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    model.eval()
    with torch.no_grad():
        logits = model(**features).logits.squeeze(-1)

    scores = torch.sigmoid(logits).tolist()

    ranked = sorted(
        [{"answer": ans, "score": float(score)} for ans, score in zip(answers, scores)],
        key=lambda x: x["score"],
        reverse=True,
    )
    return ranked[:top_k]

# Test the reranker with the current answers.
question = "what is revenue of amazon in 2023 ?"
top_results = top_k_rerank(question, answers, top_k=10)
top_results

[{'answer': "company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000\n\nAmazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.\n\nIn 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income. The year reflected a strong recovery in profitability after the weaker 2022 period. Amazon continued to focus on 